In [15]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.append(str(Path.home() / "Documents/DS217/users/jormond/open_picu"))
import pandas as pd
import inventory as I
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
rbh_dir = base / "RBH"

rbh = I.inventory_site(rbh_dir)   # add max_rows=2_000_000 for a first pass
rbh["summary"]

ai_picu_airway.csv: 615369 rows x 8 cols
ai_picu_bg.csv: 555193 rows x 29 cols
ai_picu_biochemistry.csv: 149154 rows x 38 cols
ai_picu_coagulation.csv: 124556 rows x 14 cols
ai_picu_demographics.csv: 14799 rows x 8 cols
ai_picu_full_blood_count.csv: 129719 rows x 20 cols
ai_picu_infusion.csv: 731258 rows x 15 cols
ai_picu_io.csv: 2588228 rows x 43 cols
ai_picu_meds.csv: 331165 rows x 16 cols
ai_picu_outcomes.csv: 14797 rows x 3 cols
ai_picu_piccanet.csv: 4796 rows x 21 cols
ai_picu_piccanet_old.csv: 7928 rows x 21 cols
ai_picu_sedation.csv: 1193315 rows x 15 cols
ai_picu_sedation_old.csv: 484111 rows x 15 cols
ai_picu_temp.csv: 921045 rows x 10 cols
ai_picu_vent.csv: 1705956 rows x 39 cols
ai_picu_vitals.csv: 1957204 rows x 19 cols


,file,size MB,rows,columns,duplicate rows,encoding
0,ai_picu_airway.csv,57.0,615369,8,0,utf-8
1,ai_picu_bg.csv,102.7,555193,29,0,utf-8
2,ai_picu_biochemistry.csv,23.3,149154,38,0,utf-8
3,ai_picu_coagulation.csv,13.4,124556,14,0,utf-8
4,ai_picu_demographics.csv,1.2,14799,8,0,utf-8
5,ai_picu_full_blood_count.csv,18.0,129719,20,0,utf-8
6,ai_picu_infusion.csv,115.5,731258,15,0,utf-8
7,ai_picu_io.csv,375.4,2588228,43,0,utf-8
8,ai_picu_meds.csv,51.0,331165,16,0,utf-8
9,ai_picu_outcomes.csv,0.4,14797,3,0,utf-8


In [17]:
v = pd.read_csv(rbh_dir/"ai_picu_vitals.csv",
                usecols=["studyID","PAS_AD_RK","chartTime","WARD_IN_dttm","WARD_OUT_dttm",
                         "intv_hr","Day","LOS","clinicalUnit"])
for c in ["chartTime","WARD_IN_dttm","WARD_OUT_dttm"]:
    v[c] = I.parse_dates(v[c])[0]

hrs = (v["chartTime"] - v["WARD_IN_dttm"]).dt.total_seconds() / 3600
print("intv_hr minus hours since WARD_IN:\n", (v["intv_hr"] - hrs).describe())
stay = v.drop_duplicates("studyID")
los_h = (stay["WARD_OUT_dttm"] - stay["WARD_IN_dttm"]).dt.total_seconds() / 3600
print("LOS / (WARD_OUT - WARD_IN in hours):\n", (stay["LOS"] / los_h).describe())
print("WARD_OUT before WARD_IN:", int((los_h < 0).sum()))
print(v["chartTime"].dt.year.value_counts().sort_index())

intv_hr minus hours since WARD_IN:
 count    1.949287e+06
mean     2.184196e-01
std      4.262505e-01
min     -9.998103e-01
25%      0.000000e+00
50%      2.333333e-01
75%      5.000000e-01
max      9.975000e-01
dtype: float64
LOS / (WARD_OUT - WARD_IN in hours):
 count    14786.000000
mean         0.992587
std          0.034941
min          0.000000
25%          0.995248
50%          1.000000
75%          1.000000
max          1.549720
dtype: float64
WARD_OUT before WARD_IN: 21
chartTime
2009.0      3797
2010.0     92520
2011.0    128306
2012.0    145503
2013.0    154014
2014.0    170378
2015.0    175109
2016.0    179867
2017.0    185212
2018.0    176793
2019.0    155149
2020.0     88181
2021.0    120349
2022.0    141984
2023.0     32125
Name: count, dtype: int64


In [18]:
g = v.groupby("PAS_AD_RK").agg(n_study=("studyID","nunique"),
                               n_in=("WARD_IN_dttm","nunique"),
                               n_out=("WARD_OUT_dttm","nunique"))
multi = g[g.n_study > 1]
print(len(multi), "admissions with more than one studyID")
print("distinct WARD_IN per such admission:\n", multi["n_in"].value_counts().head())
print("distinct WARD_OUT per such admission:\n", multi["n_out"].value_counts().head())

s = (v.groupby(["PAS_AD_RK","studyID"])["chartTime"].agg(["min","max"])
       .reset_index().sort_values(["PAS_AD_RK","min"]))
s["prev_max"] = s.groupby("PAS_AD_RK")["max"].shift()
print("studyIDs starting before the previous one ended:", int((s["min"] < s["prev_max"]).sum()))
print("gap to previous studyID (hours):\n",
      ((s["min"] - s["prev_max"]).dt.total_seconds() / 3600).describe())

1139 admissions with more than one studyID
distinct WARD_IN per such admission:
 n_in
2    887
3    183
4     43
5     12
6      7
Name: count, dtype: int64
distinct WARD_OUT per such admission:
 n_out
2    869
3    170
4     40
1     36
5     12
Name: count, dtype: int64
studyIDs starting before the previous one ended: 63
gap to previous studyID (hours):
 count     578.000000
mean      301.555613
std       925.309564
min     -5749.916667
25%        22.262500
50%       116.500000
75%       368.895833
max      4558.500000
dtype: float64


In [19]:
has_picu = v["clinicalUnit"].eq("RBH-PICU").groupby(v["studyID"]).any()
pn_new = set(pd.read_csv(rbh_dir/"ai_picu_piccanet.csv", usecols=["studyID"])["studyID"])
pn_old = set(pd.read_csv(rbh_dir/"ai_picu_piccanet_old.csv", usecols=["studyID"])["studyID"])
print(pd.crosstab(has_picu.rename("any PICU row"),
                  has_picu.index.isin(pn_new), rownames=["any PICU row"], colnames=["in PICANet new"]))
print("in PICANet old:", has_picu.index.isin(pn_old).sum())
print("PICANet new IDs with no PICU row:", len(pn_new - set(has_picu[has_picu].index)))

in PICANet new  False  True 
any PICU row                
False            8022    112
True             1981   4682
in PICANet old: 4837
PICANet new IDs with no PICU row: 113


In [20]:
d = pd.read_csv(rbh_dir/"ai_picu_demographics.csv")
dup = d[d["studyID"].duplicated(keep=False)]
print(len(dup), "rows;", dup["studyID"].nunique(), "studyIDs")
print("columns that differ within a studyID:\n", dup.groupby("studyID").nunique().max())

4 rows; 2 studyIDs
columns that differ within a studyID:
 PATIENT_RK      1
Age_mth         1
Gender          1
Ethnicity       1
Weight_kg       2
PICUadm_dttm    1
PAS_AD_RK       2
dtype: int64


In [21]:
d = pd.read_csv(rbh_dir/"ai_picu_demographics.csv")
parts = d["studyID"].str.split(":", expand=True)
print(parts[0].value_counts())
mid, last = parts[1].astype("int64"), parts[2].astype("int64")
print("middle == PATIENT_RK:", (mid == d["PATIENT_RK"]).mean())
print("middle == PAS_AD_RK:", (mid == d["PAS_AD_RK"]).mean())
t = I.parse_dates(d["PICUadm_dttm"])[0]
for name, epoch in [("SAS", "1960-01-01"), ("unix", "1970-01-01")]:
    ref = (t - pd.Timestamp(epoch)).dt.total_seconds()
    print(name, "last part minus PICUadm_dttm (s):\n", (last - ref).abs().describe())

0
NND    11837
NIC     2962
Name: count, dtype: int64
middle == PATIENT_RK: 1.0
middle == PAS_AD_RK: 0.0
SAS last part minus PICUadm_dttm (s):
 count    14799.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
dtype: float64
unix last part minus PICUadm_dttm (s):
 count        14799.0
mean     315619200.0
std              0.0
min      315619200.0
25%      315619200.0
50%      315619200.0
75%      315619200.0
max      315619200.0
dtype: float64


In [22]:
n_rows = v.groupby("studyID")["chartTime"].count()
print("studyIDs with no timestamped vitals rows:", int((n_rows == 0).sum()))

ad = v.drop_duplicates("studyID").set_index("studyID")["PAS_AD_RK"]
picu_ids = has_picu[has_picu].index
adm = pd.DataFrame({"PAS_AD_RK": ad,
                    "picu": ad.index.isin(picu_ids),
                    "picanet": ad.index.isin(pn_new)})
a = adm.groupby("PAS_AD_RK")[["picu", "picanet"]].any()
print(pd.crosstab(a["picu"], a["picanet"], rownames=["any PICU row"], colnames=["in PICANet new"]))

studyIDs with no timestamped vitals rows: 7916
in PICANet new  False  True 
any PICU row                
False            7085    102
True             1613   4484


In [23]:
d["prefix"]  = d["studyID"].str.split(":").str[0]
d["empty"]   = d["studyID"].map(n_rows).eq(0)
d["has_picu"] = d["studyID"].map(has_picu)
d["picanet"] = d["studyID"].isin(pn_new)
d["year"]    = I.parse_dates(d["PICUadm_dttm"])[0].dt.year

print(d.groupby("prefix")[["empty", "has_picu", "picanet"]].agg(["sum", "mean"]).round(2))
print(pd.crosstab(d["year"], d["empty"]))
print(d.groupby("empty")[["Age_mth", "Weight_kg"]].describe().T)

       empty       has_picu       picanet      
         sum  mean      sum  mean     sum  mean
prefix                                         
NIC        0  0.00     2962  1.00    2816  0.95
NND     7917  0.67     3702  0.31    1979  0.17
empty  False  True 
year               
1990       0     13
1991       0     63
1992       0     43
1993       0    328
1994       0    407
1995       0    461
1996       0    415
1997       0    424
1998       0    381
1999       0    384
2000       0    392
2001       0    398
2002       0    456
2003       0    473
2004       0    471
2005       0    460
2006       0    529
2007       0    583
2008       0    512
2009      95    523
2010     532    147
2011     614      4
2012     562     11
2013     523      3
2014     550      2
2015     545      3
2016     564      4
2017     544      2
2018     535      3
2019     491      3
2020     340      6
2021     436      5
2022     443      6
2023     108      2
empty                  False        True

In [24]:
d[["studyID", "prefix", "empty", "has_picu", "picanet", "year"]].to_csv(
    open_picu_dir/"RBH_cohort_flags.csv", index=False)

In [25]:
sed = pd.read_csv(rbh_dir/"ai_picu_sedation.csv",
                  usecols=lambda c: c.lower() in ("studyid", "patient_rk", "pas_ad_rk"))
sed.columns = sed.columns.str.lower()
ids = sed.drop_duplicates()
out = ids[~ids["studyid"].isin(d["studyID"])]

print("unique studyIDs:", ids["studyid"].nunique(),
      "| not in cohort:", out["studyid"].nunique())
print("  with missing patient_rk:", out.loc[out["patient_rk"].isna(), "studyid"].nunique())
print("  with patient_rk in cohort:", out.loc[out["patient_rk"].isin(d["PATIENT_RK"]), "studyid"].nunique())
print("  with pas_ad_rk in cohort:", out.loc[out["pas_ad_rk"].isin(d["PAS_AD_RK"]), "studyid"].nunique())
print("  prefixes:\n", out["studyid"].str.split(":").str[0].value_counts())
print("rows with missing patient_rk overall:", round(sed["patient_rk"].isna().mean(), 3))

unique studyIDs: 7576 | not in cohort: 3155
  with missing patient_rk: 0
  with patient_rk in cohort: 3023
  with pas_ad_rk in cohort: 2975
  prefixes:
 studyid
NND    3155
Name: count, dtype: int64
rows with missing patient_rk overall: 0.0


In [26]:
out = out.copy()
out["t"] = pd.Timestamp("1960-01-01") + pd.to_timedelta(
    out["studyid"].str.split(":").str[2].astype("int64"), unit="s")
print(out["t"].dt.year.value_counts().sort_index())

d["t"] = I.parse_dates(d["PICUadm_dttm"])[0]
m = out.merge(d[["PAS_AD_RK", "t"]], left_on="pas_ad_rk", right_on="PAS_AD_RK",
              suffixes=("", "_coh"))
m["dh"] = (m["t"] - m["t_coh"]).dt.total_seconds() / 3600
print("hours to nearest cohort studyID in the same admission:\n",
      m.groupby("studyid")["dh"].apply(lambda x: x.abs().min()).describe())

t
2012      9
2013    375
2014    383
2015    369
2016    348
2017    288
2018    315
2019    263
2020    175
2021    216
2022    241
2023    173
Name: count, dtype: int64
hours to nearest cohort studyID in the same admission:
 count    2975.000000
mean       64.413675
std       315.448882
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max      6554.890833
Name: dh, dtype: float64


In [27]:
mm = out.merge(d[["studyID", "PAS_AD_RK", "t"]], left_on="pas_ad_rk", right_on="PAS_AD_RK",
               suffixes=("", "_coh"))
same = mm[(mm["t"] - mm["t_coh"]).abs() < pd.Timedelta(seconds=1)]
print(same["studyid"].nunique(), "out-of-cohort studyIDs with a cohort studyID at the identical time")
print(same["studyID"].str.split(":").str[0].value_counts())   # prefix of the cohort twin
print("same patient part:",
      (same["studyid"].str.split(":").str[1] == same["studyID"].str.split(":").str[1]).mean())

2473 out-of-cohort studyIDs with a cohort studyID at the identical time
studyID
NIC    2473
Name: count, dtype: int64
same patient part: 1.0


In [28]:
nz = d[~d["empty"]]
key = nz["PATIENT_RK"].astype(str) + "|" + nz["PICUadm_dttm"].astype(str)
print("non-empty studyIDs:", nz["studyID"].nunique(), "| distinct stays (patient + time):", key.nunique())
print("prefixes per stay:\n", nz.groupby(key)["prefix"].nunique().value_counts())

sed_ids = ids["studyid"].drop_duplicates()
pref = sed_ids.str.split(":").str[0]
print(pd.crosstab(pref, sed_ids.isin(d["studyID"]), colnames=["in demographics"]))

non-empty studyIDs: 6881 | distinct stays (patient + time): 6876
prefixes per stay:
 prefix
1    6871
2       5
Name: count, dtype: int64
in demographics  False  True 
studyid                      
NIC                  0   2950
NND               3155   1471


In [29]:
sed = pd.read_csv(rbh_dir/"ai_picu_sedation.csv",
                  usecols=lambda c: c.lower() in ("studyid", "charttime", "drug", "dose"))
sed.columns = sed.columns.str.lower()
p = sed["studyid"].str.split(":", expand=True)
sed["prefix"], sed["stay"] = p[0], p[1] + ":" + p[2]

both = sed.groupby("stay")["prefix"].nunique()
both = both[both == 2].index
print("stays present under both prefixes in sedation:", len(both))

sb = sed[sed["stay"].isin(both)]
n = sb.groupby(["stay", "prefix"]).size().unstack()
print("rows per copy:\n", n.describe())
print("stays with equal row counts in both copies:", round((n["NIC"] == n["NND"]).mean(), 3))

cols = ["stay", "charttime", "drug", "dose"]
a = sb.loc[sb.prefix == "NIC", cols].drop_duplicates()
b = sb.loc[sb.prefix == "NND", cols].drop_duplicates()
print("identical records in both copies:", len(a.merge(b, on=cols)), "of", len(a), "(NIC) and", len(b), "(NND)")

stays present under both prefixes in sedation: 2478
rows per copy:
 prefix          NIC          NND
count   2478.000000  2478.000000
mean     141.154964   134.506457
std      397.530997   390.690714
min        1.000000     1.000000
25%       21.000000    20.000000
50%       38.000000    37.000000
75%      101.750000    96.000000
max     7112.000000  7112.000000
stays with equal row counts in both copies: 0.948
identical records in both copies: 324126 of 349744 (NIC) and 333268 (NND)


In [30]:
for f in sorted(rbh_dir.glob("*.csv")):
    ids = pd.read_csv(f, usecols=lambda c: c.lower() == "studyid")
    if ids.empty:
        continue
    u = ids.iloc[:, 0].dropna().drop_duplicates().str.split(":", expand=True)
    stay = u[1] + ":" + u[2]
    n_both = int((u.groupby(stay)[0].nunique() == 2).sum())
    print(f"{f.name}: {len(u)} studyIDs | {stay.nunique()} stays | {n_both} under both prefixes")

ai_picu_airway.csv: 14797 studyIDs | 14792 stays | 5 under both prefixes
ai_picu_bg.csv: 14797 studyIDs | 14792 stays | 5 under both prefixes
ai_picu_biochemistry.csv: 14797 studyIDs | 14792 stays | 5 under both prefixes
ai_picu_coagulation.csv: 14797 studyIDs | 14792 stays | 5 under both prefixes
ai_picu_demographics.csv: 14797 studyIDs | 14792 stays | 5 under both prefixes
ai_picu_full_blood_count.csv: 14797 studyIDs | 14792 stays | 5 under both prefixes
ai_picu_infusion.csv: 2884 studyIDs | 2882 stays | 2 under both prefixes
ai_picu_io.csv: 14797 studyIDs | 14792 stays | 5 under both prefixes
ai_picu_meds.csv: 5109 studyIDs | 5104 stays | 5 under both prefixes
ai_picu_outcomes.csv: 14797 studyIDs | 14792 stays | 5 under both prefixes
ai_picu_piccanet.csv: 4795 studyIDs | 4789 stays | 6 under both prefixes
ai_picu_piccanet_old.csv: 7926 studyIDs | 5502 stays | 2424 under both prefixes
ai_picu_sedation.csv: 7576 studyIDs | 5098 stays | 2478 under both prefixes
ai_picu_sedation_old.csv

In [31]:
def stays(f):
    ids = pd.read_csv(f, usecols=lambda c: c.lower() == "studyid").iloc[:, 0].dropna().drop_duplicates()
    p = ids.str.split(":", expand=True)
    return set(p[1] + ":" + p[2])

new = stays(rbh_dir/"ai_picu_piccanet.csv")
old = stays(rbh_dir/"ai_picu_piccanet_old.csv")
coh = stays(rbh_dir/"ai_picu_demographics.csv")
print("new:", len(new), "old:", len(old), "| both:", len(new & old),
      "| new only:", len(new - old), "| old only:", len(old - new))
print("old only, in cohort:", len((old - new) & coh), "| not in cohort:", len((old - new) - coh))
print("share of new in cohort:", round(len(new & coh) / len(new), 3))

new: 4789 old: 5502 | both: 4737 | new only: 52 | old only: 765
old only, in cohort: 96 | not in cohort: 669
share of new in cohort: 1.0


In [32]:
def years(keys):
    secs = pd.Series([int(k.split(":")[1]) for k in keys])
    return (pd.Timestamp("1960-01-01") + pd.to_timedelta(secs, unit="s")).dt.year

for name, keys in [("old only, in cohort", (old - new) & coh),
                   ("old only, not in cohort", (old - new) - coh),
                   ("new only", new - old)]:
    print(name, len(keys)); print(years(keys).value_counts().sort_index().to_string())

old only, in cohort 96
2012    13
2021     1
2023    82
old only, not in cohort 669
2013     60
2014     53
2015     45
2016     57
2017     28
2018     38
2019     28
2020     18
2021     75
2022     86
2023    181
new only 52
2014     6
2015     3
2016     6
2017    10
2018     4
2019     5
2020    16
2021     1
2022     1


In [33]:
pn = pd.read_csv(rbh_dir/"ai_picu_piccanet.csv", usecols=["studyID"])["studyID"].drop_duplicates()
has_data = n_rows.reindex(pn).fillna(0) > 0
print("PICANet-new studyIDs:", len(pn),
      "| with vitals rows:", int(has_data.sum()),
      "| with a PICU row:", int(has_picu.reindex(pn).fillna(False).sum()),
      "| no vitals at all:", int((~has_data).sum()))

secs = pn[~has_data.values].str.split(":").str[2].astype("int64")
print("no-vitals PICANet studyIDs by admission year:")
print((pd.Timestamp("1960-01-01") + pd.to_timedelta(secs, unit="s")).dt.year.value_counts().sort_index().to_string())

PICANet-new studyIDs: 4795 | with vitals rows: 4770 | with a PICU row: 4682 | no vitals at all: 25
no-vitals PICANet studyIDs by admission year:
studyID
2013    4
2014    2
2015    3
2016    3
2017    2
2018    1
2019    3
2020    3
2021    2
2022    2


In [34]:
vit = n_rows[n_rows > 0].index
rest = pd.Index(vit).difference(pn)
print(len(rest), "studyIDs with vitals not in PICANet-new | with a PICU row:",
      int(has_picu.reindex(rest).sum()))
print(pd.Series(rest.str.split(":").str[0]).value_counts())
secs = pd.Series(rest.str.split(":").str[2].astype("int64"))
print((pd.Timestamp("1960-01-01") + pd.to_timedelta(secs, unit="s")).dt.year.value_counts().sort_index().to_string())

2111 studyIDs with vitals not in PICANet-new | with a PICU row: 1981
studyID
NND    1965
NIC     146
Name: count, dtype: int64
studyID
2009     95
2010    532
2011    614
2012    562
2013     23
2014     33
2015     23
2016     22
2017      9
2018      9
2019     10
2020     12
2021     28
2022     31
2023    108


In [35]:
tw = same["studyid"].unique()
rest = m[~m["studyid"].isin(tw)]
g = rest.groupby("studyid")["dh"].apply(lambda x: x.abs().min())
print(len(g), "studyIDs; hours to nearest cohort studyID in the same admission:")
print(g.describe())
print((g < 24).sum(), "are within 24 h")

502 studyIDs; hours to nearest cohort studyID in the same admission:
count     502.000000
mean      381.734428
std       685.067987
min         0.016667
25%        17.195764
50%       133.083333
75%       423.375000
max      6554.890833
Name: dh, dtype: float64
161 are within 24 h


In [10]:
# 1. ID hierarchy
demo = pd.read_csv(rbh_dir / "ai_picu_demographics.csv")
print(demo.groupby("PAS_AD_RK")["studyID"].nunique().value_counts().sort_index())
print(demo.groupby("PATIENT_RK")["studyID"].nunique().describe())
print(demo[demo.duplicated("studyID", keep=False)].shape)

# 2. PICANet old vs new
pn, pno = (pd.read_csv(rbh_dir / f) for f in ["ai_picu_piccanet.csv", "ai_picu_piccanet_old.csv"])
s_new, s_old, s_all = set(pn.studyID), set(pno.studyID), set(demo.studyID)
print(len(s_new & s_old), len(s_new - s_old), len(s_old - s_new), len(s_new - s_all), len(s_old - s_all))

# 3. Timestamp formats (first rows)
pd.read_csv(rbh_dir / "ai_picu_vitals.csv", nrows=5)[["chartTime", "WARD_IN_dttm", "WARD_OUT_dttm", "Day", "intv_hr"]]

# 4. Ventilation modes and charting interval
vent = pd.read_csv(rbh_dir / "ai_picu_vent.csv", usecols=["studyID", "intv_hr", "Ventilation_Mode"])
print(vent["Ventilation_Mode"].value_counts(dropna=False))
gap = vent.sort_values(["studyID", "intv_hr"]).groupby("studyID")["intv_hr"].diff()
print(gap.describe(percentiles=[.1, .5, .9, .99]))

studyID
1     12146
2       888
3       183
4        43
5        12
6         7
7         3
8         1
11        1
13        1
Name: count, dtype: int64
count    10019.000000
mean         1.476894
std          1.075666
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         25.000000
Name: studyID, dtype: float64
(4, 8)
4743 52 3183 1 3089
Ventilation_Mode
BiPAP                    507660
NaN                      501449
Self Vent                307031
CPAP                     154159
Nasal High Flow           58407
PC-BIPAP                  53387
CPAP+Pressure Support     31134
BILEVEL                   27261
SIMV                      18483
SIMV + PS                  9995
VC-SIMV                    8961
HFOV                       8070
High Flow                  7675
SPN-CPAP/PS                5430
NIV                        2522
PCV                        1306
ASB                        1296
PC-SIMV                     569
IPPV                

In [11]:
rbh["profiles"]["ai_picu_vitals.csv"]

,column,dtype,missing %,unique,detail
0,studyID,str,0.0,14797,avg length 22; first value shape aaa:ddddddd:dddddddddd
1,PATIENT_RK,int64,0.0,10019,min 84; p1 53533; median 1.04646e+06; p99 1.30342e+06; max 1.32331e+06
2,Weight_kg,float64,2.9,591,min 0.6; p1 1.3; median 5.4; p99 80; max 175
3,Day,float64,0.4,389,min 0; p1 0; median 6; p99 182; max 1717
4,intv_hr,float64,0.4,9163,min 0; p1 0; median 151; p99 4363; max 41225
5,chartTime,str,0.4,703832,avg length 22; first value shape ddaaadddd:dd:dd:dd.ddd
6,clinicalUnit,str,0.4,9,RBH-PICU: 1615795; RBH-PHDU: 195389; RBH-Theatre: 108087; RBH-Cath: 29553; RBH-Reginald-Wilson: 355; RBH-Recovery: 72; RBH-AICU: 19; RBH-Alex: 13; RBH-EICU/EHDU: 4
7,HR,float64,10.4,300,min 0; p1 59; median 130; p99 184; max 300
8,Underlying_Cardiac_Rhythm,str,97.8,361,avg length 4; first value shape aaaaa
9,NIBP_s,float64,56.0,212,min 0; p1 45; median 81; p99 129; max 256


In [12]:
I.shared_columns(rbh)

,ai_picu_airway.csv,ai_picu_bg.csv,ai_picu_biochemistry.csv,ai_picu_coagulation.csv,ai_picu_demographics.csv,ai_picu_full_blood_count.csv,ai_picu_infusion.csv,ai_picu_io.csv,ai_picu_meds.csv,ai_picu_outcomes.csv,ai_picu_piccanet.csv,ai_picu_piccanet_old.csv,ai_picu_sedation.csv,ai_picu_sedation_old.csv,ai_picu_temp.csv,ai_picu_vent.csv,ai_picu_vitals.csv
addate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2456.0,2540.0,NaN,NaN,NaN,NaN,NaN
addatetime,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4383.0,4673.0,NaN,NaN,NaN,NaN,NaN
adtime,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,608.0,632.0,NaN,NaN,NaN,NaN,NaN
agedays,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1778.0,1799.0,NaN,NaN,NaN,NaN,NaN
agemonths,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,204.0,204.0,NaN,NaN,NaN,NaN,NaN
ageyears,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.0,18.0,NaN,NaN,NaN,NaN,NaN
calcium,NaN,313.0,304.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
charttime,116879.0,NaN,NaN,NaN,NaN,NaN,95031.0,144850.0,101290.0,NaN,NaN,NaN,160530.0,88391.0,295280.0,481990.0,703832.0
clinicalunit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,9.0,9.0
corr_calcium,NaN,261.0,299.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
I.id_overlap(rbh)

,column,file,unique ids,% in base
0,studyid,ai_picu_airway.csv,14797,100.0
1,studyid,ai_picu_bg.csv,14797,100.0
2,studyid,ai_picu_biochemistry.csv,14797,100.0
3,studyid,ai_picu_coagulation.csv,14797,100.0
4,studyid,ai_picu_full_blood_count.csv,14797,100.0
5,studyid,ai_picu_infusion.csv,2884,100.0
6,studyid,ai_picu_io.csv,14797,100.0
7,studyid,ai_picu_meds.csv,5109,100.0
8,studyid,ai_picu_outcomes.csv,14797,100.0
9,studyid,ai_picu_piccanet.csv,4795,100.0


In [14]:
I.write_report(rbh, "RBH", open_picu_dir / "RBH_inventory.md")